In [6]:
import Pkg; Pkg.activate(@__DIR__); Pkg.add("PyPlot"); Pkg.add("ForwardDiff"); Pkg.add("Ipopt"); Pkg.instantiate()

  Activating project at `/workspaces/lecture-notebooks/Lecture 13`
   Resolving package versions...
  No Changes to `/workspaces/lecture-notebooks/Lecture 13/Project.toml`
  No Changes to `/workspaces/lecture-notebooks/Lecture 13/Manifest.toml`
   Resolving package versions...
  No Changes to `/workspaces/lecture-notebooks/Lecture 13/Project.toml`
  No Changes to `/workspaces/lecture-notebooks/Lecture 13/Manifest.toml`
   Resolving package versions...
   Installed XML2_jll ────────────── v2.13.8+0
   Installed SPRAL_jll ───────────── v2025.5.20+0
   Installed OpenBLAS32_jll ──────── v0.3.24+0
   Installed Ipopt ───────────────── v1.11.0
   Installed Ipopt_jll ───────────── v300.1400.1900+0
   Installed MUMPS_seq_jll ───────── v500.800.100+0
   Installed Xorg_libpciaccess_jll ─ v0.18.1+0
   Installed Hwloc_jll ───────────── v2.12.2+0
   Installed METIS_jll ───────────── v5.1.3+0
   Installed ASL_jll ─────────────── v0.1.3+0
    Updating `/workspaces/lecture-notebooks/Lecture 13/Project.

In [7]:
using LinearAlgebra
using PyPlot
using ForwardDiff
using RobotZoo
const RZ = RobotZoo
using RobotDynamics
using Ipopt
using MathOptInterface
const MOI = MathOptInterface;

In [8]:
#Pendulum Dynamics
a = RobotZoo.Acrobot()
h = 0.1 #10 Hz

0.1

In [ ]:
Nx = 4     # number of state
Nu = 1     # number of controls
Tfinal = 5.0 # final time
Nt = Int(Tfinal/h)+1   # number of time steps
thist = Array(range(0,h*(Nt-1), step=h));
n_nlp = (Nx+Nu)*Nt # number of decision variables
m_nlp = Nx*(Nt+1) # number of constraints

208

In [10]:
function dircol_dynamics(x1,u1,x2,u2)
    #Hermite-Simpson integration with first-order hold on u
    f1 = RZ.dynamics(a, x1, u1)
    f2 = RZ.dynamics(a, x2, u2)
    xm = 0.5*(x1 + x2) + (h/8.0)*(f1 - f2)
    um = 0.5*(u1 + u2)
    ẋm = (-3/(2.0*h))*(x1 - x2) - 0.25*(f1 + f2)
    fm = RZ.dynamics(a, xm, um)
    return fm - ẋm
end

dircol_dynamics (generic function with 1 method)

In [11]:
# Cost weights
Q = Diagonal([1.0*ones(2); 1.0*ones(2)]);
R = 0.1;

In [12]:
function stage_cost(x,u)
    return 0.5*((x-xgoal)'*Q*(x-xgoal)) + 0.5*u'*R*u
end

stage_cost (generic function with 1 method)

In [13]:
function cost(ztraj)
    z = reshape(ztraj,Nx+Nu,Nt)
    J = 0.0
    for k = 1:Nt
        J += stage_cost(z[1:Nx,k],z[(Nx+1):(Nx+Nu),k])
    end
    return J
end

cost (generic function with 1 method)

In [14]:
function dynamics_constraint!(c,ztraj)
    d = reshape(c,Nx,Nt-1)
    z = reshape(ztraj,Nx+Nu,Nt)
    for k = 1:(Nt-1)
        x1 = z[1:Nx,k]
        u1 = z[(Nx+1):(Nx+Nu),k]
        x2 = z[1:Nx,k+1]
        u2 = z[(Nx+1):(Nx+Nu),k+1]
        d[:,k] = dircol_dynamics(x1,u1,x2,u2)
    end
    return nothing
end

dynamics_constraint! (generic function with 1 method)

In [15]:
function con!(c,ztraj)
    z = reshape(ztraj,Nx+Nu,Nt)
    c[1:Nx] .= z[1:Nx,1] - x0
    @views dynamics_constraint!(c[(Nx+1):(end-Nx)],ztraj)# 利用视图防止函数内部对内存空间的拷贝
    c[(end-Nx+1):end] .= z[1:Nx,end] - xgoal
end

con! (generic function with 1 method)

In [34]:
struct ProblemMOI <: MOI.AbstractNLPEvaluator
    n_nlp::Int
    m_nlp::Int
    idx_ineq
    obj_grad::Bool
    con_jac::Bool
    sparsity_jac
    sparsity_hess
    primal_bounds
    constraint_bounds
    hessian_lagrangian::Bool
end

function ProblemMOI(n_nlp,m_nlp;
        idx_ineq=(1:0),
        obj_grad=true,
        con_jac=true,
        sparsity_jac=sparsity_jacobian(n_nlp,m_nlp),
        sparsity_hess=sparsity_hessian(n_nlp,m_nlp),
        primal_bounds=primal_bounds(n_nlp),
        constraint_bounds=constraint_bounds(m_nlp,idx_ineq=idx_ineq),
        hessian_lagrangian=false)

    ProblemMOI(n_nlp,m_nlp,
        idx_ineq,
        obj_grad,
        con_jac,
        sparsity_jac,
        sparsity_hess,
        primal_bounds,
        constraint_bounds,
        hessian_lagrangian)
end

function primal_bounds(n)
    x_l = -Inf*ones(n)
    x_u = Inf*ones(n)
    return x_l, x_u
end

function constraint_bounds(m; idx_ineq=(1:0))
    c_l = zeros(m)
    c_l[idx_ineq] .= -Inf

    c_u = zeros(m)
    return c_l, c_u
end

function row_col!(row,col,r,c)
    for cc in c
        for rr in r
            push!(row,convert(Int,rr))
            push!(col,convert(Int,cc))
        end
    end
    return row, col
end

function sparsity_jacobian(n,m)

    row = []
    col = []

    r = 1:m
    c = 1:n

    row_col!(row,col,r,c)

    return collect(Tuple{Int64, Int64}, zip(row,col))
end

function sparsity_hessian(n,m)

    row = []
    col = []

    r = 1:m
    c = 1:n

    row_col!(row,col,r,c)

    return collect(Tuple{Int64, Int64}, zip(row,col))
end

function MOI.eval_objective(prob::MOI.AbstractNLPEvaluator, x)
    cost(x)
end

function MOI.eval_objective_gradient(prob::MOI.AbstractNLPEvaluator, grad_f, x)
    ForwardDiff.gradient!(grad_f,cost,x)
    return nothing
end

function MOI.eval_constraint(prob::MOI.AbstractNLPEvaluator,g,x)
    con!(g,x)
    return nothing
end

function MOI.eval_constraint_jacobian(prob::MOI.AbstractNLPEvaluator, jac, x)
    ForwardDiff.jacobian!(reshape(jac,prob.m_nlp,prob.n_nlp), con!, zeros(prob.m_nlp), x)
    return nothing
end

function MOI.features_available(prob::MOI.AbstractNLPEvaluator)
    return [:Grad, :Jac]
end

MOI.initialize(prob::MOI.AbstractNLPEvaluator, features) = nothing
MOI.jacobian_structure(prob::MOI.AbstractNLPEvaluator) = prob.sparsity_jac

function solve(x0,prob::MOI.AbstractNLPEvaluator;
        tol=1.0e-4,c_tol=1.0e-4,max_iter=1000)
    x_l, x_u = prob.primal_bounds
    c_l, c_u = prob.constraint_bounds

    nlp_bounds = MOI.NLPBoundsPair.(c_l,c_u)
    block_data = MOI.NLPBlockData(nlp_bounds,prob,true)

    solver = Ipopt.Optimizer()
    solver.options["max_iter"] = max_iter
    solver.options["tol"] = tol
    solver.options["constr_viol_tol"] = c_tol

    x = MOI.add_variables(solver,prob.n_nlp)

    for i = 1:prob.n_nlp
        MOI.add_constraint(solver, x[i], MOI.LessThan(x_u[i]))
        MOI.add_constraint(solver, x[i], MOI.GreaterThan(x_l[i]))
        MOI.set(solver, MOI.VariablePrimalStart(), x[i], x0[i])
    end

    # Solve the problem
    MOI.set(solver, MOI.NLPBlock(), block_data)
    MOI.set(solver, MOI.ObjectiveSense(), MOI.MIN_SENSE)
    MOI.optimize!(solver)

    # Get the solution
    res = MOI.get(solver, MOI.VariablePrimal(), x)

    return res
end

solve (generic function with 1 method)

In [24]:
#Initial and goal states
x0 = [-pi/2; 0; 0; 0]
xgoal = [pi/2; 0; 0; 0]

4-element Vector{Float64}:
 1.5707963267948966
 0.0
 0.0
 0.0

In [35]:
#Guess
xguess = kron(ones(Nt)', x0)
#xguess[1,:] .= xguess[1,:] + 0.1*randn(Nt) #运行一遍之后解注释探讨利用不满足动力学的初值对算法收敛的影响
uguess = zeros(Nt)'
z0 = reshape([xguess; uguess],(Nx+Nu)*Nt,1);

In [36]:
prob = ProblemMOI(n_nlp,m_nlp)

ProblemMOI(255, 208, 1:0, true, true, [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1)  …  (199, 255), (200, 255), (201, 255), (202, 255), (203, 255), (204, 255), (205, 255), (206, 255), (207, 255), (208, 255)], [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1)  …  (199, 255), (200, 255), (201, 255), (202, 255), (203, 255), (204, 255), (205, 255), (206, 255), (207, 255), (208, 255)], ([-Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf  …  -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf, -Inf], [Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf  …  Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf]), ([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), false)

In [ ]:
z_sol = solve(z0,prob) # solve

In [ ]:
ztraj = reshape(z_sol,Nx+Nu,Nt)
xtraj = ztraj[1:Nx,:]
utraj = ztraj[(Nx+1):(Nx+Nu),:];

In [ ]:
plot(thist,xtraj[1,:])
plot(thist,xtraj[2,:])
plot(thist,xguess[1,:])
plot(thist,xguess[2,:])

In [ ]:
plot(thist,xtraj[3,:])
plot(thist,xtraj[4,:])

In [ ]:
plot(thist,utraj[1,:])

In [ ]:
# Acrobot (doublependulum)
import MeshCat as mc
using Colors

function RotX(alpha)
    c, s = cos(alpha), sin(alpha)
    [1 0 0; 0 c -s; 0 s  c]
end

function create_acrobot!(vis, color=colorant"blue", thick=0.05)
    l1,l2 = [1.,1.]
    hinge = mc.Cylinder(mc.Point(-0.05,0,0), mc.Point(0.05,0,0), 0.05)
    dim1  = mc.Vec(thick, thick, l1)
    link1 = mc.HyperRectangle(mc.Vec(-thick/2,-thick/2,0),dim1)
    dim2  = mc.Vec(thick, thick, l2)
    link2 = mc.HyperRectangle(mc.Vec(-thick/2,-thick/2,0),dim2)
    mat1 = mc.MeshPhongMaterial(color=colorant"grey")
    mat2 = mc.MeshPhongMaterial(color=color)
    mc.setobject!(vis["base"], hinge, mat1) 
    mc.setobject!(vis["link1"], link1, mat2) 
    mc.setobject!(vis["link1","joint"], hinge, mat1) 
    mc.setobject!(vis["link1","link2"], link2, mat2) 
    mc.settransform!(vis["link1","link2"], mc.Translation(0,0,l1))
    mc.settransform!(vis["link1","joint"], mc.Translation(0,0,l1))
end

function update_acro_pose!(vis, x)
    l1, l2 = [1, 1.]
    mc.settransform!(vis["robot","link1"], mc.LinearMap(RotX(x[1]-pi/2)))
    mc.settransform!(vis["robot","link1","link2"], mc.compose(mc.Translation(0,0,l1), mc.LinearMap(RotX(x[2]))))
end

function animate_acrobot(X, dt)
    vis = mc.Visualizer()
    create_acrobot!(vis["robot"])
    anim = mc.Animation(vis, fps=floor(Int,1/dt))
    for k = 1:length(X)
        mc.atframe(anim, k) do
            update_acro_pose!(vis,X[k])
        end
    end
    mc.setanimation!(vis, anim)
    return mc.render(vis)
end

In [ ]:
X1 = [Vector(x) for x in eachcol(xtraj)];
animate_acrobot(X1, h)